In [5]:
import pandas as pd
import os
import glob

# ==========================================
# 1. 設定 & パス定義 (ここを確実に合わせる)
# ==========================================
RUN_ID = "20251130_153500"
ROOT_BASE = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"

# 各ファイルの格納場所を定義
DIR_THESIS = os.path.join(ROOT_BASE, "Thesis_Figures", f"run_{RUN_ID}")
DIR_LABELS = os.path.join(ROOT_BASE, "sub", "04_summary_STEP3.2_signlessCorr", f"run_{RUN_ID}", "samples")
DIR_DETAIL = os.path.join(ROOT_BASE, "Thesis_Figures", f"run_{RUN_ID}", "paper_4.2.4.4_OHFP", "reverse", "analysis_csv")

# 出力先 (Thesis_Figures直下)
OUTPUT_DIR = DIR_THESIS

print(f"Target Run ID: {RUN_ID}")

# ==========================================
# 2. ファイル検索関数
# ==========================================
def find_file_in_dir(directory, pattern):
    if not os.path.exists(directory):
        print(f"[WARN] Directory not found: {directory}")
        return None
    
    # パターン検索
    search_path = os.path.join(directory, pattern)
    files = glob.glob(search_path)
    
    if files:
        # 見つかった最初のファイルを返す
        print(f"  Found: {os.path.basename(files[0])}")
        return files[0]
    else:
        print(f"  [ERROR] Not found: {pattern} in {directory}")
        return None

# ==========================================
# 3. ファイル特定
# ==========================================
print("\nLocating files...")

# 1. ThesisData (MDS + PCE)
file_thesis_A = find_file_in_dir(DIR_THESIS, '*ThesisData_A_OH_plus_others_MDS_Targets*.csv')
file_thesis_B = find_file_in_dir(DIR_THESIS, '*ThesisData_B_FP_plus_others_MDS_Targets*.csv')

# 2. Cluster Labels (Matrix)
file_labels_A = find_file_in_dir(DIR_LABELS, '*cluster_labels_matrix_samples_A_OH_plus_others*.csv')
file_labels_B = find_file_in_dir(DIR_LABELS, '*cluster_labels_matrix_samples_B_FP_plus_others*.csv')

# 3. Detail Files (Cluster Contents)
file_detail_OH_k50 = find_file_in_dir(DIR_DETAIL, 'Detail_OH_Clusters_linear_top3_k50.csv')
file_detail_OH_k10 = find_file_in_dir(DIR_DETAIL, 'Detail_OH_Clusters_linear_top3_k10.csv')
file_detail_FP_k50 = find_file_in_dir(DIR_DETAIL, 'Detail_FP_Clusters_linear_top3_k50.csv')
file_detail_FP_k10 = find_file_in_dir(DIR_DETAIL, 'Detail_FP_Clusters_linear_top3_k10.csv')

# ==========================================
# 4. データ読み込み & 結合
# ==========================================
def load_and_merge(f_thesis, f_labels):
    if not f_thesis or not f_labels: return None
    try:
        df_t = pd.read_csv(f_thesis)
        if 'Unnamed: 0' in df_t.columns:
            df_t.rename(columns={'Unnamed: 0': 'ID'}, inplace=True)
        df_l = pd.read_csv(f_labels)
        return pd.merge(df_t, df_l, on='ID', how='inner')
    except Exception as e:
        print(f"[ERROR] Merge failed: {e}")
        return None

def load_detail(path):
    return pd.read_csv(path) if path and os.path.exists(path) else None

print("\nLoading and Merging Data...")
df_A = load_and_merge(file_thesis_A, file_labels_A)
df_B = load_and_merge(file_thesis_B, file_labels_B)

det_OH_k50 = load_detail(file_detail_OH_k50)
det_OH_k10 = load_detail(file_detail_OH_k10)
det_FP_k50 = load_detail(file_detail_FP_k50)
det_FP_k10 = load_detail(file_detail_FP_k10)

# ==========================================
# 5. 集計ロジック
# ==========================================
def get_stats(df, col_k, cluster_id, detail_df, type_name):
    if df is None: return None
    subset = df[df[col_k] == cluster_id]
    if subset.empty: return None
        
    pce_data = subset['PCEmax']
    
    desc = "N/A"
    if detail_df is not None:
        row = detail_df[detail_df['Cluster_ID'] == cluster_id]
        if not row.empty:
            col_comp = 'Major_Material' if type_name == 'OH' else 'Major_Fragment'
            raw_name = str(row[col_comp].values[0])
            desc = raw_name.replace("SMILESsname", "").replace("p1M_", "").replace("nM_", "").replace("inM_", "")
    
    return {
        'Count': len(subset),
        'Mean_PCE': pce_data.mean(),
        'Std_Dev': pce_data.std(),
        'Max_PCE': pce_data.max(),
        'Major_Component': desc
    }

# ==========================================
# 6. テーブル作成実行
# ==========================================

# --- A. Table 4.x (Sweet Spot Summary) ---
if df_A is not None and df_B is not None:
    print("\nGenerating Table 4.x (Sweet Spot Summary)...")
    rows = []
    
    # 1. OH k=10 Best
    best_id = df_A.groupby('linear_top3_k10')['PCEmax'].mean().idxmax()
    st = get_stats(df_A, 'linear_top3_k10', best_id, det_OH_k10, 'OH')
    rows.append(['OH (Material)', 10, best_id, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 2. OH k=50 Best
    best_id_50 = 19
    st = get_stats(df_A, 'linear_top3_k50', best_id_50, det_OH_k50, 'OH')
    rows.append(['OH (Material)', 50, best_id_50, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 3. FP k=10 Best
    best_id = df_B.groupby('linear_top3_k10')['PCEmax'].mean().idxmax()
    st = get_stats(df_B, 'linear_top3_k10', best_id, det_FP_k10, 'FP')
    rows.append(['FP (Structure)', 10, best_id, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    # 4. FP k=50 Best
    best_id_50_fp = 18
    st = get_stats(df_B, 'linear_top3_k50', best_id_50_fp, det_FP_k50, 'FP')
    rows.append(['FP (Structure)', 50, best_id_50_fp, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    cols = ['Dataset', 'Resolution (k)', 'Cluster ID', 'N', 'Mean PCE (%)', 'Std Dev', 'Max PCE (%)', 'Major Component']
    df_main = pd.DataFrame(rows, columns=cols)
    
    out_path = os.path.join(OUTPUT_DIR, 'Table_4_x_Sweet_Spot_Summary.csv')
    df_main.to_csv(out_path, index=False)
    print(f"  [SAVE] {out_path}")
    print(df_main)

# --- B. Appendix Tables (Full Ranking) ---
def generate_full_ranking(df, detail_df, type_name, out_name):
    print(f"\nGenerating Full Ranking for {type_name}...")
    col_k = 'linear_top3_k50'
    cids = sorted(df[col_k].unique())
    rr = []
    for cid in cids:
        st = get_stats(df, col_k, cid, detail_df, type_name)
        rr.append([cid, st['Count'], st['Mean_PCE'], st['Std_Dev'], st['Max_PCE'], st['Major_Component']])
    
    df_rank = pd.DataFrame(rr, columns=['Cluster ID', 'N', 'Mean PCE (%)', 'Std Dev', 'Max PCE (%)', 'Major Component'])
    df_rank = df_rank.sort_values('Mean PCE (%)', ascending=False)
    
    out_path = os.path.join(OUTPUT_DIR, out_name)
    df_rank.to_csv(out_path, index=False)
    print(f"  [SAVE] {out_path}")

if df_A is not None and det_OH_k50 is not None:
    generate_full_ranking(df_A, det_OH_k50, 'OH', 'Table_Appendix_OH_All_Clusters_k50.csv')

if df_B is not None and det_FP_k50 is not None:
    generate_full_ranking(df_B, det_FP_k50, 'FP', 'Table_Appendix_FP_All_Clusters_k50.csv')

print("\n✅ All Tables Generated Successfully!")

Target Run ID: 20251130_153500

Locating files...
  Found: ThesisData_A_OH_plus_others_MDS_Targets.csv
  Found: ThesisData_B_FP_plus_others_MDS_Targets.csv
  Found: cluster_labels_matrix_samples_A_OH_plus_others_20251130_153500.csv
  Found: cluster_labels_matrix_samples_B_FP_plus_others_20251130_153500.csv
  Found: Detail_OH_Clusters_linear_top3_k50.csv
  Found: Detail_OH_Clusters_linear_top3_k10.csv
  Found: Detail_FP_Clusters_linear_top3_k50.csv
  Found: Detail_FP_Clusters_linear_top3_k10.csv

Loading and Merging Data...

Generating Table 4.x (Sweet Spot Summary)...
  [SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/Thesis_Figures/run_20251130_153500/Table_4_x_Sweet_Spot_Summary.csv
          Dataset  Resolution (k)  Cluster ID    N  Mean PCE (%)   Std Dev  \
0   OH (Material)              10           8   18      9.179444  0.847713   
1   OH (Material)              50          19   11      9.255455 